# 2a — $\mathcal{L}_{X_f}\omega = 0$

**Problem.** Let $(M, \omega)$ be a symplectic manifold with $d\omega = 0$, and let $\pi = \omega^{-1} \in \Gamma(\wedge^2 TM)$ be the associated Poisson bivector. For $f \in C^\infty(M)$, define the Hamiltonian vector field $X_f$ by

$$
\iota_{X_f}\omega = df, \qquad \{f, g\} := \pi(df, dg).
$$

**Goal (a).** Prove $\mathcal{L}_{X_f}\omega = 0$.

---

**Sign convention.** Bu notebook, soru metninin konvansiyonunu takip eder ($\iota_{X_f}\omega = df$). Library'deki `HamiltonianVectorField` default'u $\iota_{X_f}\omega = -df$ kullanır; bu notebook ona dokunmuyor — aşağıdaki `X_f` düz bir `Derivation` olarak inşa ediliyor ve problem-given iki aksiyom inline `Definition` olarak engine'e yükleniyor.

## Strateji

Cartan'ın sihirli formülü $\mathcal{L}_X = d\circ\iota_X + \iota_X\circ d$ hedefi iki parçaya ayırır:

$$
\mathcal{L}_{X_f}\omega = d(\iota_{X_f}\omega) + \iota_{X_f}(d\omega).
$$

İki parça da verilen aksiyomlarla kapanır:

| Parça | Aksiyom | Sonuç |
|---|---|---|
| $d(\iota_{X_f}\omega)$ | $\iota_{X_f}\omega = df$ | $d(df) = 0$ (çünkü $d^2 = 0$) |
| $\iota_{X_f}(d\omega)$ | $d\omega = 0$ (symplectic) | $\iota_{X_f}(0) = 0$ |

Toplam: $0 + 0 = 0$.

Sistemimiz bu zinciri tek `ExpandAndSimplify.prove(...)` çağrısıyla kapatır: Cartan açılımı `LieDerivativeCartanDefinition`'dan, `d^2 = 0` `DSquaredZeroDefinition`'dan, `ι_X(0) = 0` Leibniz katmanından, iki aksiyom ise notebook içinde tanımlayacağımız iki küçük `Definition` sınıfından gelir.

In [1]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Kurulum — semboller ve grading

- $f$ bir fonksiyon (0-form, `Graded(degree=0)`).
- $\omega$ bir 2-form (`Graded(degree=2)`) — intrinsik p-form yapısı yok; sistem onu sadece graded-commutative çarpım altında doğru işaretle taşıyan bir sembol olarak biliyor.
- $X_f$ derece-0 bir `Derivation` (vektör alanı).

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d, ExteriorDerivative
from jacopy.calculus.interior import interior, InteriorProduct
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Expr, Integer, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

reg = PropertyRegistry()

f = Symbol("f")
reg.declare(f, Graded(degree=0))

omega = Symbol("ω")
reg.declare(omega, Graded(degree=2))

X_f = Derivation("X_f", degree=0)

print("f     :", f, "  degree =", reg.get(f, Graded).degree)
print("ω     :", omega, "  degree =", reg.get(omega, Graded).degree)
print("X_f   :", X_f, "  degree =", X_f.degree)

f     : f   degree = 0
ω     : ω   degree = 2
X_f   : X_f   degree = 0


## 2. Verilen iki aksiyom

Problem metni iki eşitliği veri olarak sunuyor — bunları engine'in bildiği yerleşik kuralların yanına iki ufak `Definition` olarak koyarız.

**Aksiyom 1 — Symplectic closedness.** $d\omega = 0$.

**Aksiyom 2 — Hamiltonian defining relation.** $\iota_{X_f}\omega = df$.

Her `Definition` iki method'dan ibaret: `matches(expr)` LHS kalıbının tutup tutmadığını söyler, `rewrite(expr)` RHS'ı üretir.

In [3]:
class DOmegaClosed(Definition):
    """dω = 0 — the symplectic form is closed."""

    name = "dω = 0 (symplectic)"

    def matches(self, expr):
        return (
            isinstance(expr, Act)
            and isinstance(expr.op, ExteriorDerivative)
            and expr.op == d
            and expr.arg == omega
        )

    def rewrite(self, expr):
        return Integer(0)


class IotaXfOmegaIsDf(Definition):
    """ι_{X_f} ω = df — the Hamiltonian defining relation."""

    name = "ι_{X_f} ω = df"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_f:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, f)


print("axiom 1:", DOmegaClosed.name)
print("axiom 2:", IotaXfOmegaIsDf.name)

axiom 1: dω = 0 (symplectic)
axiom 2: ι_{X_f} ω = df


## 3. Engine

`default_engine` şunları getirir:
- `LieDerivativeCartanDefinition` — $\mathcal{L}_X(\omega) \to (d\circ\iota_X + \iota_X\circ d)(\omega)$
- `DSquaredZeroDefinition` — $d(d(x)) \to 0$ (`d_squared_mode="axiom"`)
- `IotaOnZeroFormDefinition` — $\iota_X(0) \to 0$
- + diğer yerleşik kurallar (Leibniz, Act-over-Sum, ι-kare, ι(df) pairing)

Buraya iki problem aksiyomunu `register(...)` ile ekleriz.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(DOmegaClosed())
engine.register(IotaXfOmegaIsDf())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 10 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - dω = 0 (symplectic)
 - ι_{X_f} ω = df


## 4. Hedef ve ispat

LHS $= \mathcal{L}_{X_f}\omega$ olarak inşa edilir ve `Integer(0)` ile eşitlenir. `ExpandAndSimplify` stratejisi obstruction $=$ LHS $-$ RHS üzerinde engine'i fix-point'e kadar çalıştırır, ardından simplify pipeline'ını uygular; residual $0$'a düşerse zinciri döndürür.

In [5]:
L_Xf = lie_derivative(X_f)  # cartan-mode default
residual = Act(L_Xf, omega)

print("LHS:", residual)
print("RHS:", Integer(0))

chain = ExpandAndSimplify().prove(
    residual, Integer(0), registry=reg, engine=engine
)
print(f"\nKAPANDI — {len(chain)} adım.")

LHS: L_X_f(ω)
RHS: 0

KAPANDI — 7 adım.


## 5. İspat zinciri — LaTeX

In [6]:
from jacopy.display.jupyter import display_chain

display_chain(chain)

\begin{align*}
L_{X_f}\!\left(\omega\right) &\to \left(d \, \iota_{X_f}\right)\!\left(\omega\right) + \left(\iota_{X_f} \, d\right)\!\left(\omega\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
\left(\left(d \, \iota_{X_f}\right)\!\left(\omega\right) + \left(\iota_{X_f} \, d\right)\!\left(\omega\right)\right) - 0 &\to \left(d\!\left(\iota_{X_f}\!\left(\omega\right)\right) + \iota_{X_f}\!\left(d\!\left(\omega\right)\right)\right) - 0 && \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\iota_{X_f}\!\left(\omega\right) &\to d\!\left(f\right) && \text{[\ensuremath{\iota}\_{X\_f} \ensuremath{\omega} = df]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_{X\_f} \ensuremath{\omega} = df} \\
d\!\left(d\!\left(f\right)\right) &\to 0 && \text{[d² = 0]\,(axiom)}\;\text{--- apply axiom: d² = 0} \\
d\!\left(\omega\right) &\to 0 && \text{[d\ensuremath{\omega} = 0 (symplectic)]\,(axiom)}\;\text{--- apply axiom: d\ensuremath{\omega} = 0 (symplectic)} \\
\iota_{X_f}\!\left(0\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\left(0 + 0\right) - 0 &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{align*}

## 6. Adım adım — hangi kural ne yaptı?

Her adımın `rule` alanı o adımda hangi Definition'ın fire ettiğini söyler. Aşağıdaki döküm, zincirin strateji içindeki akışını izler:

1. **Cartan magic** — `L_Xf(ω)` → `d(ι_Xf(ω)) + ι_Xf(d(ω))` (yerleşik).
2. **product-rule** — kompozisyon `d∘ι_Xf` açılıyor, residual `Sum(lhs, -rhs)` şeklini alıyor.
3. **ι_{X_f} ω = df** — ilk parçaya problem aksiyomu.
4. **d² = 0** — `d(d(f))` sıfır (yerleşik).
5. **dω = 0** — ikinci parçaya problem aksiyomu.
6. **ι_X(0) = 0** — kalan ι sıfıra düşüyor (yerleşik, 0-form özel hali).
7. **simplify** — `0 + 0 + (-0) → 0`, kanonik form kapanışı.

In [7]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" ↦  {step.after}")
    print()

[1] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_f(ω)
 ↦  ((d * ι_X_f)(ω) + (ι_X_f * d)(ω))

[2] product-rule 
    (((d * ι_X_f)(ω) + (ι_X_f * d)(ω)) + (-0))
 ↦  ((d(ι_X_f(ω)) + ι_X_f(d(ω))) + (-0))

[3] ι_{X_f} ω = df [axiom]
    ι_X_f(ω)
 ↦  d(f)

[4] d² = 0 [axiom]
    d(d(f))
 ↦  0

[5] dω = 0 (symplectic) [axiom]
    d(ω)
 ↦  0

[6] ι_X(f) = 0 on 0-forms [axiom]
    ι_X_f(0)
 ↦  0

[7] simplify 
    ((0 + 0) + (-0))
 ↦  0



## Sonuç

$\boxed{\mathcal{L}_{X_f}\omega = 0}$ kapandı. Kullandığımız sistem kalemleri:

- **Yerleşik**: Cartan magic açılımı, Leibniz/linearity (`product_rule`), `d² = 0`, `ι_X(0) = 0`, simplify pipeline.
- **Problem-özel**: iki `Definition` (`dω = 0`, `ι_{X_f}ω = df`). Her biri 3–4 satır; engine'e `register(...)` ile takılıyor.

Bu kalıp, "ders kitabı bir eşitliği iki-üç verilen aksiyomla kapat" şeklindeki soruların neredeyse tamamına uyar — aksiyomları problem metninden `Definition` sınıflarına çevirmek yeterli.